### Agisoft Drone Pipeline
* This pipeline needs to be ran on an instance with metashape installed and have an active licence

In [0]:
import datetime as dt
import Metashape
import os
import shutil
import requests

In [0]:
run_date = dt.datetime.now().strftime("%Y-%m-%d")

# This is the Batch File from Jorge. (click to view)
<?xml version="1.0" encoding="UTF-8"?>
<batchjobs version="2.1.1">
  <job name="AlignPhotos" enabled="false" target="all">
    <keypoint_limit>60000</keypoint_limit>
    <keypoint_limit_per_mpx>4000</keypoint_limit_per_mpx>
    <mask_tiepoints>false</mask_tiepoints>
    <tiepoint_limit>0</tiepoint_limit>
  </job>
  <job name="OptimizeCameras" enabled="false" target="all">
    <fit_b1>true</fit_b1>
    <fit_b2>true</fit_b2>
    <fit_k4>true</fit_k4>
  </job>
  <job name="BuildPointCloud" enabled="false" target="all">
    <downscale>2</downscale>
    <filter_mode>3</filter_mode>
    <reuse_depth>true</reuse_depth>
  </job>
  <job name="DetectMarkers" enabled="false" target="all">
    <tolerance>80</tolerance>
  </job>
  <job name="LocateReflectancePanels" enabled="false" target="all"/>
  <job name="CalibrateReflectance" enabled="false" target="all">
    <use_sun_sensor>true</use_sun_sensor>
  </job>
  <job name="BuildDem" enabled="false" target="all">
    <downscale>2</downscale>
    <filter_mode>3</filter_mode>
    <reuse_depth>true</reuse_depth>
  </job>
  <job name="BuildOrthomosaic" enabled="false" target="all"/>
</batchjobs>

In [0]:
# # Step 1: Start Metashape in offscreen mode (non-blocking)
# metashape_process = subprocess.Popen(
#     ["/opt/agisoft/metashape-pro/metashape.sh", "-platform", "offscreen"],
#     stdout=subprocess.DEVNULL,
#     stderr=subprocess.DEVNULL
# )

# time.sleep(15)  # Wait until metsashape is ready

print(Metashape.app.version)

 Agisoft License Key: ([REDACTED])


In [0]:
dbutils.widgets.text("flight_metadata_paths", "")
raw_paths = dbutils.widgets.get("flight_metadata_paths")

if not raw_paths:
    dbutils.notebook.exit("Error: No se recibió ninguna ruta de vuelo del trigger. Apagando nodo.")

flights_list = raw_paths.split(',')

flights_list = [p.replace("dbfs:/", "/dbfs/") for p in flights_list]

print(f" Recibida la orden de procesar un bloque de {len(flights_list)} vuelos.")

Metashape.app.gpu_mask = 1
Metashape.app.cpu_enable = False

for json_path in flights_list:
    print("\n" + "=" * 70)
    print(f" STARTING MISSION: {json_path}")
    print("=" * 70)

    PROJECT_DIR = os.path.dirname(json_path)

    OUTPUTS_DIR = PROJECT_DIR

    base_raw_dir = os.path.join(PROJECT_DIR,
                                "raw_data")
    sensor_type = ""
    if os.path.exists(os.path.join(base_raw_dir,
                                   "multi-spec")):
        RAW_IMG_DIR = os.path.join(base_raw_dir,
                                   "multi-spec")
        sensor_type = "MS"
        print("  Directorio de imágenes detectado: multi-spec")
    elif os.path.exists(os.path.join(base_raw_dir, "rgb")):
        RAW_IMG_DIR = os.path.join(base_raw_dir, "rgb")
        sensor_type = "RGB"
        print("  Directorio de imágenes detectado: rgb")
    else:
        print(f"  Error: No se encontró la subcarpeta 'rgb' ni 'multi-spec' en {base_raw_dir}")
        continue  # Salta al siguiente vuelo de la lista

    flight_name = os.path.basename(PROJECT_DIR)
    local_tmp = f"/tmp/metashape_{flight_name}"
    input_images_local = f"{local_tmp}/local_images"
    outputs_local = f"{local_tmp}/outputs"

    GCP = False
    has_reflectance_panels = True

    if os.path.exists(local_tmp):
        shutil.rmtree(local_tmp)

    os.makedirs(input_images_local, exist_ok=True)
    os.makedirs(outputs_local, exist_ok=True)

    if os.path.exists(RAW_IMG_DIR):
        pictures = [f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(('.tif', '.jpg', '.jpeg'))]
        print(f" Copy {len(pictures)} images into local SSD...")
        for img in pictures:
            shutil.copy2(os.path.join(RAW_IMG_DIR, img), os.path.join(input_images_local, img))
    else:
        print(f" Error: There is no Raw_Data folder in {PROJECT_DIR}. skipping flight.")
        continue

    photos = [os.path.join(input_images_local, f) for f in os.listdir(input_images_local) if
              f.lower().endswith(('.tif', '.jpg', '.jpeg'))]
    print(f" Starting metashape with {len(photos)} images...")

    try:
        agisoft_license_key = dbutils.secrets.get(scope="agisoft_creds", key="agisoft_license_key")
        Metashape.license.activate(agisoft_license_key)

        doc = Metashape.Document()
        doc.save(local_tmp + '/project.psx')
        chunk = doc.addChunk()

        print('Adding Markers if there are any')
        if GCP:

            if 'marker_gdf' in locals() and len(marker_gdf) > 0:
                for idx, row in marker_gdf.iterrows():
                    marker = chunk.addMarker()
                    marker.label = row['label']
                    marker.reference.location = Metashape.Vector([row.x, row.y, row.h])
                    marker.reference.enabled = True

        print('Adding the photos')
        chunk.addPhotos(photos)
        doc.save()

        print('Match the photos')
        chunk.matchPhotos(keypoint_limit=60000,
                          tiepoint_limit=0,
                          keypoint_limit_per_mpx=4000,
                          generic_preselection=True,
                          reference_preselection=True)
        doc.save()

        print('Align the Cameras')
        chunk.alignCameras()
        doc.save()
        chunk.updateTransform()

        print('Optimize the cameras')
        chunk.optimizeCameras(fit_b1=True, fit_b2=True, fit_k4=True)
        doc.save()

        print('Build the Depth Maps')
        chunk.buildDepthMaps(downscale=2, filter_mode=Metashape.NoFiltering, max_neighbors=16)
        doc.save()

        print('Detect the markers and import the reference points')
        if GCP:
            chunk.detectMarkers(target_type=Metashape.TargetType.CircularTarget, tolerance=80, filter_mask=False,
                                maximum_residual=15)
            chunk.importReference()
        else:
            print('No Marker')

        print('Locate the Reflectance Panels')
        if has_reflectance_panels:
            chunk.locateReflectancePanels()
            doc.save()

        print('Calibrate the Reflectance Panels')
        chunk.calibrateReflectance()
        doc.save()

        print('Build Point Cloud')
        chunk.buildPointCloud()
        doc.save()

        print('Build DEM')
        chunk.buildDem(source_data=Metashape.PointCloudData)
        doc.save()

        print('Build Orthomosaic')
        chunk.buildOrthomosaic(surface_data=Metashape.ElevationData)
        doc.save()

        print('Exporting results to local temporary folder...')
        chunk.exportReport(outputs_local + '/report.pdf')

        if chunk.model:
            chunk.exportModel(outputs_local + '/model.obj')

        if chunk.elevation:
            chunk.exportRaster(outputs_local + '/DEM.tif', source_data=Metashape.ElevationData)

        if chunk.orthomosaic:
            if sensor_type == "MS":
                chunk.exportRaster(outputs_local + '/MS.tif', source_data=Metashape.OrthomosaicData)
                print("Multispectral orthomosaic exported as MS.tif")
            else:
                chunk.exportRaster(outputs_local + '/RGB.tif', source_data=Metashape.OrthomosaicData)
                print("Standard orthomosaic exported as RGB.tif")

        print(f'Processing finished, results saved to {outputs_local}.')

        print(f'Uploading final results to the flight folder: {OUTPUTS_DIR}')
        generated_files = os.listdir(outputs_local)
        for file in generated_files:
            origen = os.path.join(outputs_local, file)
            destino = os.path.join(OUTPUTS_DIR, file)
            shutil.copy2(origen, destino)

        print(f' Mission {flight_name} Completed and files successfully uploaded.')

    except Exception as e:

        print(f" An error occurred in the processing of {flight_name}: {e}")

    finally:

        Metashape.license.deactivate()

        if os.path.exists(local_tmp):
            shutil.rmtree(local_tmp)
            print(f" Cleaning the local SSD environment ({local_tmp}) completed.")

print("\n The entire batch of pending missions has been processed.")

🚀 Iniciando procesamiento en disco local...
SaveProject: path = /tmp/metashape/project.psx
saved project in 0.011274 sec
LoadProject: path = /tmp/metashape/project.psx
loaded project in 0.000589 sec
Adding Markers if there are any

License activated



Adding the photos
AddPhotos
SaveProject: path = /tmp/metashape/project.psx
saved project in 0.104118 sec
Match the photos
MatchPhotos: downscale = 1, generic_preselection = on, reference_preselection = on, filter_mask = off, mask_tiepoints = on, filter_stationary_points = on, keypoint_limit = 60000, keypoint_limit_per_mpx = 4000, tiepoint_limit = 0, guided_matching = off, exclude_corners = off
saved matching data in 0.004903 sec
saved object list in 0.000397 sec
scheduled 31 keypoint detection groups
saved keypoint partition in 0.000279 sec
groups: 1799 1799 1799 1799 1799 1799 1799 1799 1791
1820 of 612 used (297.386%)
scheduled 9 keypoint matching groups
saved matching partition in 0.000477 sec
loaded object list in 4.1e-05 sec
loaded keypoint partition in 2.1e-05 sec
loaded matching data in 1.8e-05 sec
Found 1 GPUs in 0.066601 sec (CUDA: 0.039209 sec, OpenCL: 0.000373 sec, Vulkan: 0.026994 sec)


Can't load OpenCL library


Using device: NVIDIA L40S, 142 compute units, free memory: 45155/45596 MB, compute capability 8.9
  driver/runtime CUDA: 12020/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
  got device properties in 0.000206 sec, free memory in 36.4768 sec
[GPU] photo 0: 60000 points
[GPU] photo 31: 60000 points
[GPU] photo 62: 60000 points
[GPU] photo 93: 60000 points
[GPU] photo 124: 60000 points
[GPU] photo 155: 60000 points
[GPU] photo 186: 60000 points
[GPU] photo 217: 60000 points
[GPU] photo 248: 60000 points
[GPU] photo 279: 60000 points
[GPU] photo 310: 60000 points
[GPU] photo 341: 60000 points
[GPU] photo 372: 60000 points
[GPU] photo 403: 60000 points
[GPU] photo 434: 60000 points
[GPU] photo 465: 60000 points
[GPU] photo 496: 60000 points
[GPU] photo 527: 60000 points
[GPU] photo 558: 60000 points
[GPU] photo 589: 60000 points
points detected in 41.407 sec
loaded object list in 4.3e-05 sec
loaded keypoint partition in 2.1e-05 sec
loaded matching data in 1.8e-05 s

...

Can't load OpenCL library


Using device 'NVIDIA L40S' in concurrent. (2 times)
Camera 191 (16 neighbs) level #1/3 filtering: 84% good (34% of speckles) + 5% norm (66% of speckles) - 0% speckles + 1% bad + 10% empty (51% inliers support + 0% inliers intersects + 0% inliers doesn't reach + 0% inliers no depth + 48% outliers support + 1% outliers intersects + 2% outliers doesn't reach + 0% inliers occludes + 1% outliers occludes)
Camera 191 (16 neighbs) level #2/3 filtering: 83% good (9% of speckles) + 11% norm (91% of speckles) - 0% speckles + 2% bad + 4% empty (0% inliers support + 0% inliers intersects + 0% inliers doesn't reach + 0% inliers no depth + 99% outliers support + 1% outliers intersects + 1% outliers doesn't reach + 0% inliers occludes + 1% outliers occludes)
Camera 191 (16 neighbs) level #3/3 filtering: 86% good (9% of speckles) + 10% norm (91% of speckles) - 0% speckles + 1% bad + 2% empty (0% inliers support + 0% inliers intersects + 0% inliers doesn't reach + 0% inliers no depth + 100% outliers su

Can't load OpenCL library


Using device 'NVIDIA L40S' in concurrent. (2 times)
Camera 5 (1 neighbs) level #1/3 filtering: 0% good (0% of speckles) + 2% norm (100% of speckles) - 0% speckles + 7% bad + 91% empty (23% inliers support + 6% inliers intersects + 31% inliers doesn't reach + 40% inliers no depth + 0% outliers support + 0% outliers intersects + 0% outliers doesn't reach + 13% inliers occludes + 0% outliers occludes)
Camera 5 (1 neighbs) level #2/3 filtering: 0% good (0% of speckles) + 3% norm (100% of speckles) - 0% speckles + 10% bad + 87% empty (25% inliers support + 6% inliers intersects + 27% inliers doesn't reach + 43% inliers no depth + 0% outliers support + 0% outliers intersects + 0% outliers doesn't reach + 53% inliers occludes + 0% outliers occludes)
Camera 5 (1 neighbs) level #3/3 filtering: 0% good (0% of speckles) + 3% norm (100% of speckles) - 0% speckles + 4% bad + 92% empty (47% inliers support + 4% inliers intersects + 18% inliers doesn't reach + 32% inliers no depth + 0% outliers suppo

License deactivated
No nodelocked license found
No license server found


In [0]:
JOB_3_ID = 162763862056473  #Change

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

url = f"{host}/api/2.1/jobs/run-now"
headers = {"Authorization": f"Bearer {token}"}
data = {"job_id": JOB_3_ID}

response = requests.post(url,
                         headers=headers,
                         json=data)

if response.status_code == 200:
    print(" Trigger successful! Heavy processing Job has been started.")
else:
    print(f" Error triggering Job: {response.text}")